Part 1: Ingesting Documents
1.	Read the given data.
a.	Observe that the given data is a database file.
b.	Go through the README.txt to understand what is there inside the database.
c.	Take care of decoding the files inside the database.
d.	If you have limited compute resources, you can take a random 30% of the data.
2.	Apply appropriate cleaning steps on subtitle documents (whatever is required)
3.	Experiment with the following to generate text vectors of subtitle documents:
a.	BOW / TFIDF to generate sparse vector representations. Note that this will only help you to build a Keyword Based Search Engine.
b.	BERT based “SentenceTransformers” to generate embeddings which encode semantic information. This can help us build a Semantic Search Engine.
4.	(Must Implement) A very important step to improve the performance: Document Chunker.
a.	Consider the challenge of embedding large documents: Information Loss.
b.	It is often not practical to embed an entire document as a single vector, particularly when dealing with long documents.
c.	Solution: Divide a large document into smaller, more manageable chunks for embedding.
d.	Another Problem: Let’s say we set the token window to be 500, then we’d expect each chunk to be just below 500 tokens. One common concern of this method is that we might accidentally cut off some important text between chunks, splitting up the context. To mitigate this, we can set overlapping windows with a specified amount of tokens to overlap so we have tokens shared between chunks.
5.	Store embeddings in a ChromaDB database. 

Part 2: Retrieving Documents
1.	Take the user's search query in audio format. This audio must be a 2 minutes recording of a TV series or Movie which is a part of the database.
2.	Preprocess the query (if required).
3.	Create query embedding.
4.	Using cosine distance, calculate the similarity score between embeddings of documents and user search query embedding.
5.	These cosine similarity scores will help in returning the most relevant candidate documents as per user’s search query.


## Project Overview: Enhancing Search Engine Relevance for Video Subtitles (Cloning Shazam)

### What is This Project About?

**Goal**: Build a search engine that finds TV show/movie subtitles based on audio recordings (similar to Shazam for music, but for subtitles).

**How It Works**:
1. **Part 1 - Ingestion**: Take subtitle files → Clean them → Split into chunks → Convert to embeddings (vectors) → Store in database
2. **Part 2 - Retrieval**: User provides audio → Convert to text → Convert to embedding → Search database → Find matching subtitles

### Key Concepts Explained

**1. What is an "Embedding"?**
- An embedding is a numerical representation of text
- It's a list of numbers (typically 1536 numbers for OpenAI's model) that captures the meaning of text
- Texts with similar meanings have similar embeddings
- We can compare embeddings using math (cosine similarity)

**2. Why Do We Clean Data?**
- Remove timestamps (00:00:00 --> 00:00:10) - not meaningful for searching
- Remove HTML tags (<i>, <b>) - not part of actual dialogue
- Remove sequence numbers - just formatting
- Result: Pure dialogue text that can be analyzed meaningfully

**3. Why Do We Chunk Documents?**
- A full movie subtitle file is too long (can be thousands of lines)
- OpenAI's embedding model works best with smaller chunks
- We create overlapping chunks so context isn't lost between chunks
- Each chunk becomes a searchable item with metadata (which subtitle it came from)

**4. What is ChromaDB?**
- A vector database (stores embeddings and their metadata)
- Optimized for similarity searches
- Fast: can search millions of embeddings in milliseconds
- We store all subtitle chunks + their embeddings here

**5. How Does Semantic Search Work?**
- Convert your search query to an embedding (same way as subtitle chunks)
- Find the embeddings most similar to your query embedding
- Return the actual subtitle text for the closest matches
- Uses cosine similarity (mathematical measure of closeness)

```python
google_embed_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
langchain_docs = [Document(page_content=chunk, metadata=metadata) for chunk in chunks]
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=google_embed_model,
    collection_name="subtitle_chunks",
    persist_directory="./chroma_db"
)
```

## Part 2: Retrieving Documents - Detailed Explanation

### Step 1: Audio to Text Conversion (Whisper)

```python
def audio_to_text(audio_path):
    import whisper
    model = whisper.load_model("base")
    result = model.transcribe(audio_path, language="en")
    transcribed_text = result['text']
    return transcribed_text
```

**What's happening:**

1. `whisper.load_model("base")` - Downloads/loads the Whisper model
   - "base" = ~140M parameters (balanced size/speed)
   - Other options: tiny, small, medium, large (larger = more accurate but slower)

2. `model.transcribe(audio_path, language="en")` - Converts audio to text
   - Input: Path to audio file (mp3, wav, m4a, etc.)
   - Processes: Sends audio to Whisper model
   - Output: Dictionary with results
   - `language="en"`: Tells Whisper to expect English

3. `result['text']` - Extracts just the transcribed text
   - Result might also contain timestamps, confidence scores (we ignore these)

**Example:**
```
Input audio file: "sample.mp3" (2 minutes of dialogue from a movie)
Output: "I love you so much. This summer was unforgettable and magical."
```

---

### Step 2: Query Preprocessing

```python
def preprocess_query(text):
    return clean_subtitle_text(text)
```

**What's happening:**

- Takes the transcribed text from audio
- Applies the SAME cleaning we did for subtitles
- Removes timestamps, HTML tags, etc.
- Result: Clean text ready for embedding

**Why same cleaning?**
- Subtitles were cleaned in the same way
- Query should be in same format as database documents
- Ensures consistent matching

**Example:**
```
Input: "I love you so much. This summer was unforgettable and magical."
Output: "I love you so much. This summer was unforgettable and magical."
(no change since there are no timestamps/tags to remove)
```

---

### Step 3: Create Query Embedding

```python
def create_query_embedding(query_text):
    cleaned_query = preprocess_query(query_text)
    query_embedding = openai_embed_model.embed_query(cleaned_query)
    return cleaned_query, np.array(query_embedding)
```

**What's happening:**

1. `preprocess_query()` - Clean the text (explained above)

2. `openai_embed_model.embed_query()` - Convert text to embedding
   - Input: Cleaned text string
   - Process: Sends to OpenAI API
   - Output: 1536-dimensional vector

3. `np.array()` - Converts to NumPy array for easier math operations

4. **Return**: Both the cleaned text and embedding

**Why we need the embedding:**
- ChromaDB stores embeddings (not raw text)
- To search, we must convert query to same embedding format
- We can then compare embeddings using math

**Visual representation:**
```
Query text: "I love you so much. This summer was unforgettable."
    ↓ (OpenAI Embedding)
Embedding: [0.123, -0.456, 0.789, ..., 0.321]  ← 1536 numbers
```

---

### Step 4: Semantic Search (Finding Matches)

```python
def semantic_search(query_text, k=5):
    vectorstore = Chroma(
        collection_name="subtitle_chunks",
        embedding_function=openai_embed_model,
        persist_directory="./chroma_db"
    )
    results = vectorstore.similarity_search_with_score(query_text, k=k)
    
    formatted_results = []
    for doc, score in results:
        formatted_results.append({
            'content': doc.page_content,
            'similarity_score': score,
            'subtitle_name': doc.metadata.get('subtitle_name')
        })
    return formatted_results
```

**What's happening:**

1. `Chroma()` - Connect to the vector database we created in Part 1
   - Loads all stored embeddings and metadata
   - `collection_name`: Which collection to search (we called it "subtitle_chunks")
   - `persist_directory`: Where database is saved ("./chroma_db")

2. `similarity_search_with_score()` - Search for similar chunks
   - Input: Query text (automatically gets converted to embedding)
   - Process: Compares query embedding to all stored embeddings
   - Math: Uses cosine similarity (how "similar" two vectors are)
   - `k=5`: Return top 5 most similar chunks
   - Output: List of (Document, similarity_score) pairs

3. **Similarity Score Explained:**
   - Range: 0 to 1 (or sometimes 0 to 2)
   - Higher number = MORE SIMILAR (sometimes lower, depends on metric)
   - ChromaDB uses cosine distance, so LOWER is better (0 = perfect match)

4. Loop through results and format nicely
   - Extract content (the actual subtitle text)
   - Extract score (how similar it is)
   - Extract metadata (which subtitle it came from)

**Visual Example:**
```
Query: "summer of freedom"

ChromaDB compares to all stored embeddings:
  - Chunk 1: similarity = 0.15 ✓✓✓ (BEST MATCH)
  - Chunk 2: similarity = 0.28 ✓✓
  - Chunk 3: similarity = 0.42 ✓
  - Chunk 4: similarity = 0.51
  - Chunk 5: similarity = 0.63

Returns top 5 results (already sorted by similarity)
```

---

### Step 5: Complete Pipeline (Audio → Results)

```python
def search_by_audio(audio_path, k=10):
    print("Step 1: Converting audio to text...")
    query_text = audio_to_text(audio_path)        # ← Whisper
    
    print("Step 2: Preprocessing query...")
    cleaned_query = preprocess_query(query_text)   # ← Clean
    
    print("Step 3: Searching...")
    results = semantic_search(cleaned_query, k=k)  # ← Search
    
    return {
        "original_query": query_text,
        "cleaned_query": cleaned_query,
        "results": results
    }
```

**What's happening (end-to-end):**

1. User provides audio file (2 minutes of TV show/movie)
2. Whisper transcribes it to text
3. Text is cleaned (remove any artifacts from transcription)
4. Text is converted to embedding (1536 numbers)
5. ChromaDB finds 10 most similar stored chunks
6. Returns original subtitle text + similarity scores

**Complete Flow:**
```
Audio File
    ↓ (Whisper)
Transcribed Text: "I love you so much. This summer..."
    ↓ (Preprocess)
Cleaned Text: "I love you so much. This summer..."
    ↓ (Embedding)
Embedding: [0.123, -0.456, ..., 0.321]
    ↓ (Similarity Search)
Top 10 Matching Subtitle Chunks
    ↓ (Format Results)
[
  {"subtitle": "Movie1.srt", "content": "I love you...", "score": 0.15},
  {"subtitle": "Movie2.srt", "content": "That summer...", "score": 0.28},
  ...
]
```

---

### Key Takeaways

**Part 1 (Ingestion):**
- Read subtitle files → Clean → Chunk → Embed → Store in database
- One-time process (takes time but only done once)

**Part 2 (Retrieval):**
- Fast and simple: Audio → Embed → Search database
- Uses semantic similarity (meaning-based, not keyword-based)
- Much smarter than keyword search (finds paraphrases, synonyms, etc.)

---

## Python Concepts Used (Beginner's Guide)

### 1. Functions (`def`)

```python
def clean_subtitle_text(text):
    """Docstring: explains what function does"""
    # Function body
    return cleaned_text
```

**What's happening:**
- `def` keyword creates a function (reusable block of code)
- `clean_subtitle_text` is the function name
- `(text)` is the input parameter
- `"""..."""` is documentation (what function does)
- `return` sends result back to caller
- Can be called multiple times: `clean_subtitle_text("some text")`

---

### 2. List Comprehensions (Efficient loops)

```python
# Traditional way (long):
results = []
for doc, score in search_results:
    results.append({"content": doc.page_content, "score": score})

# List comprehension (short and fast):
results = [{"content": doc.page_content, "score": score} for doc, score in search_results]
```

**What's happening:**
- Square brackets `[ ]` create a list
- `for doc, score in search_results` loops through items
- Creates new dictionary for each item
- All in one compact line!

---

### 3. Dictionaries (Storing related data)

```python
result = {
    'subtitle_id': 9180533,
    'subtitle_name': 'movie.srt',
    'similarity_score': 0.156,
    'content': 'I love you...'
}

# Accessing values:
print(result['subtitle_id'])          # ← Access by key
print(result.get('content'))          # ← Safe access (returns None if missing)
```

**What's happening:**
- Curly braces `{ }` create a dictionary
- `key: value` pairs store related information
- Access using square brackets: `dict['key']`
- `.get()` is safer (won't crash if key missing)

---

### 4. F-strings (Formatted text)

```python
name = "Alice"
score = 0.156

# Traditional way:
print("Hello " + name + " your score is " + str(score))

# F-string way (cleaner):
print(f"Hello {name} your score is {score}")

# With formatting:
print(f"Score: {score:.2f}")  # ← Shows 2 decimal places: 0.16
```

**What's happening:**
- `f"..."` prefix makes it an f-string
- `{variable}` inserts variable value into string
- `:` allows formatting options (`.2f` = 2 decimal places)

---

### 5. Regular Expressions (Pattern matching)

```python
import re

text = "Hello 123 World"

# Find pattern:
result = re.sub(r'\d+', 'XXX', text)
# Result: "Hello XXX World"

# Pattern breakdown: r'\d+' means:
# r = raw string (ignore backslashes)
# \d = any digit (0-9)
# + = one or more
```

**Common patterns:**
- `\d` = digit (0-9)
- `\w` = word character (a-z, A-Z, 0-9, _)
- `\s` = whitespace (space, tab, newline)
- `+` = one or more
- `*` = zero or more
- `^` = start of line
- `$` = end of line
- `.` = any character
- `[a-z]` = any letter a through z

---

### 6. Context Managers (`with` statement)

```python
# Without context manager (manual cleanup):
f = open('file.txt', 'r')
content = f.read()
f.close()  # ← Must remember to close!

# With context manager (automatic cleanup):
with open('file.txt', 'r') as f:
    content = f.read()
# ← File automatically closed, even if error occurs!
```

**What's happening:**
- `with` statement manages setup/cleanup automatically
- File opens → read content → automatically closes
- If error occurs, still closes properly
- Much safer and cleaner!

---

### 7. Try-Except (Error handling)

```python
try:
    result = model.transcribe(audio_path)
    print("Success!")
except Exception as e:
    print(f"Error: {e}")
    # Continue running instead of crashing
```

**What's happening:**
- `try` block: code that might fail
- `except` block: code to run if error occurs
- `Exception as e`: captures error message
- Program doesn't crash, continues running

---

### 8. Numpy Arrays (Numerical data)

```python
import numpy as np

# Create arrays:
arr = np.array([1, 2, 3, 4, 5])

# Mathematical operations:
result = arr * 2              # ← [2, 4, 6, 8, 10]
result = np.sum(arr)          # ← 15
result = np.mean(arr)         # ← 3.0

# Embeddings are numpy arrays:
embedding = np.array([0.123, -0.456, 0.789, ...])  # ← 1536 numbers
```

**Why numpy?**
- Very fast (optimized for math)
- Designed for numerical computing
- Perfect for embeddings and machine learning

---

### 9. Pandas DataFrames (Tables)

```python
import pandas as pd

# Create from database:
df = pd.read_sql_query("SELECT * FROM zipfiles", conn)

# Inspect data:
print(df.head())             # ← First 5 rows
print(df.info())             # ← Column types, non-null counts
print(df.shape)              # ← (82498, 3) = 82498 rows, 3 columns

# Access columns:
names = df['name']           # ← Get 'name' column
df['new_col'] = df['col'].apply(func)  # ← Apply function to column

# Statistics:
print(df.describe())         # ← Mean, min, max, etc.
```

**What's happening:**
- DataFrame = table (rows and columns)
- Similar to Excel spreadsheet or SQL table
- Very efficient for data manipulation
- Perfect for processing thousands of records

---

### 10. Lambda Functions (Inline functions)

```python
# Regular function:
def double(x):
    return x * 2

# Lambda function (one-liner):
double = lambda x: x * 2

# Using in apply:
df['file_content'] = df['content'].apply(lambda x: decode_method(x))
# ← Calls decode_method on each row

# Alternative with named function:
df['file_content'] = df['content'].apply(decode_method)
```

**When to use:**
- Simple one-liner functions
- Functions used only once
- Functions passed to other functions (like `apply()`)

---

### Practice Examples

**Example 1: Using functions**
```python
def greet(name):
    return f"Hello, {name}!"

result = greet("Alice")
print(result)  # ← Output: "Hello, Alice!"
```

**Example 2: Using list comprehension**
```python
numbers = [1, 2, 3, 4, 5]
squares = [x**2 for x in numbers]
print(squares)  # ← [1, 4, 9, 16, 25]
```

**Example 3: Using dictionaries + f-strings**
```python
person = {'name': 'Alice', 'age': 30, 'city': 'NYC'}
print(f"{person['name']} is {person['age']} years old")
# ← Output: "Alice is 30 years old"
```

**Example 4: Using try-except**
```python
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Can't divide by zero!")
```

---

### Tips for Learning

1. **Run code step by step**: Execute each cell individually
2. **Print intermediate results**: Use `print()` to see what's happening
3. **Read error messages**: They tell you exactly what went wrong
4. **Modify code**: Change values and see what happens
5. **Google errors**: Almost every error has been solved before!
6. **Use documentation**: Hover over functions to see help

Good luck with your learning! 🚀

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-chroma
!pip install langchain-google-genai
!pip install google-generativeai

  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.24.0
    Uninstalling zstandard-0.24.0:
      Successfully uninstalled zstandard-0.24.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.4.59
    Uninstalling langsmith-0.4.59:
      Successfully uninstalled langsmith-0.4.59
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.1.3
    Uninstalling langchain-core-1.1.3:
      Successfully uninstalled langchain-core-1.1.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.11 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.3.45 which is incompatible.
langchain-openai 0.2.12 requires openai<2.0.0,>=1.55.3, but you have openai 2.14.0 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.


   ---------------------------------------- 0.0/948.6 kB ? eta -:--:--
   ---------------------------------------- 948.6/948.6 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.14.0
    Uninstalling openai-2.14.0:
      Successfully uninstalled openai-2.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openai-agents 0.6.4 requires openai<3,>=2.9.0, but you have openai 1.109.1 which is incompatible.


  Using cached langsmith-0.2.11-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.78-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.77-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.76-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_core-0.3.75-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-0.3.74-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-0.3.73-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-0.3.72-py3-none-any.whl.metadata (5.8 kB)
INFO: This is taking longer than usual. You might 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.63 which is incompatible.


  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
   ---------------------------------------- 0.0/783.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/783.6 kB ? eta -:--:--
   ---------------------------------------- 783.6/783.6 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.2.11
    Uninstalling langsmith-0.2.11:
      Successfully uninstalled langsmith-0.2.11
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.41.1
    Uninstalling google-auth-2.41.1:
      Successfully uninstalled google-auth-2.41.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.63
    Uninstalling langchain-core-0.3.63:
      Successfully uninstalled langchain-core-0.3.63
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.49.0
    Uninstalling google-genai-1.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.3.0 which is incompatible.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.7.32 which is incompatible.
langchain-chroma 0.1.4 requires langchain-core<0.4,>=0.1.40; python_version >= "3.9", but you have langchain-core 1.3.0 which is incompatible.
langchain-community 0.3.11 requires langchain-core<0.4.0,>=0.3.24, but you have langchain-core 1.3.0 which is incompatible.
langchain-community 0.3.11 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.7.32 which is incompatible.
langchain-openai 0.2.12 requires langchain-core<0.4.0,>=0.3.21, but you have langchain-core 1.3.0 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.3.0 w

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.76.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.75.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.74.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.0-py3-none-any.whl.metadata (1.1 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.72.2-py

In [2]:
import sqlite3
import pandas as pd

In [3]:
# Read the code below and write your observation in the next cell

conn = sqlite3.connect('eng_subtitles_database.db')
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('zipfiles',)]


In [4]:
cursor.execute("PRAGMA table_info('zipfiles')")
cols = cursor.fetchall()
for col in cols:
    print(col[1])

num
name
content


In [5]:
df = pd.read_sql_query("""SELECT * FROM zipfiles""", conn)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82498 entries, 0 to 82497
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   num      82498 non-null  int64 
 1   name     82498 non-null  object
 2   content  82498 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.9+ MB


In [6]:
b_data = df.iloc[0, 2]

In [7]:
print(b_data[:500])  # Print the first 500 characters of the binary data

b"PK\x03\x04\x14\x00\x00\x00\x08\x00\x1c\xa9\x99V\x9fx\x96\xf0\x8c\x9e\x00\x00\x86\x9b\x01\x00;\x00\x00\x00The.Message.1976.REMASTERED.1080p.BluRay.x264-PiGNUS.EN.srt\xad\xbdm\x93\xdc\xc6\x91.\xfa\x9d\x11\xfc\x0f-}\xe1=\x11-\x9d\x06P\x85\x17\x9d\x8d\xd5%%[\xa4-Y>&u\x15>\xdf\xd0\xd3\x98\x19x\xfae\x0cts<\xfe\xf57\x9f'\xb3\n\xd9\xa4\xbc\xbb\xf7\xc6Fl\xacELW\xa2\xaa\x90\x95\x95\xafO\x16/_l6\xdf\xe0\xff\xea\xf5f\xb3Y}\xf5\xd5\xbf\xaf\xf4AQ\xae7Mx\xf9\xe2\xd7\xfe|s\xbf\xea\x8f\xcf\xab\x8f\xe3n8\xadN\xc7\xfdx\x1cVO\xe3\xf9~\xf5\xf3\xe3p\xfc\xea\xfd/o>\xbc\xfb\xf0\xe3\xef\xde\xbf|\xf1\xfbi\x18Vo\xa6\xd3\xd3<L\xab\xe1\x1f\xe7\xe18\x8f\xa7\xe37\xab\xd3\xbc\xdb~-\xc3\x1e\xfe\xa7<|\xf9\xe2\xe5\x8bR_[~S\xd6\xeb\xa2k\xf3k\xe5A\xb7\xeeb\xf5\xf2\xc5\xbb\xe3\xea|?\xac\x8e\xfdaX\x9dnW?\x9cvk>8\x9c\xe6\xf3\xean\xeao\xc6\xd3ev\x8f~\x1a\xa6\x9b\xf1\xf6\xb2\xff\x1a\xe4\xabD\xbe*d\x11\xa5#_U\xeb\xaa\xd9`\xa6\xa7\xc3\xea\xa7\xcb}\x7f8\xf4F\xf9\xa7a\x9e\x87\xe3\x9d\xcc\\\xdf\x07B!\x13\xaa\xd61n<!\xd9\xaf\xd0\x

In [8]:
import zipfile
import io

count = 0

def decode_method(binary_data):
    global count
    # Decompress the binary data using the zipfile module
    # print(count, end=" ")
    count += 1
    with io.BytesIO(binary_data) as f:
        with zipfile.ZipFile(f, 'r') as zip_file:
            # Assuming there's only one file in the ZIP archive
            subtitle_content = zip_file.read(zip_file.namelist()[0])
    
    # Now 'subtitle_content' should contain the extracted subtitle content
    return subtitle_content.decode('latin-1')  # Assuming the content is UTF-8 encoded text

In [9]:
df['file_content'] = df['content'].apply(decode_method)

df.head()

,num,name,content,file_content
0,9180533,the.message.(1976).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x1c\xa9\x...,"1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch an..."
1,9180583,here.comes.the.grump.s01.e09.joltin.jack.in.bo...,b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x17\xb9\x...,"1\r\n00:00:29,359 --> 00:00:32,048\r\nAh! Ther..."
2,9180592,yumis.cells.s02.e13.episode.2.13.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00L\xb9\x99V...,"1\r\n00:00:53,200 --> 00:00:56,030\r\n<i>Yumi'..."
3,9180594,yumis.cells.s02.e14.episode.2.14.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00U\xa9\x99V...,"1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch an..."
4,9180600,broker.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x001\xa9\x99V...,"ï»¿1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch..."


In [11]:
!pip install langchain-chroma

In [10]:
# Enter Google API key
from getpass import getpass
GOOGLE_API_KEY = getpass("Enter your Google API Key here: ")

In [11]:
# Setup Environment Variables
import os
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

In [12]:
# Google Embedding Model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
google_embed_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

In [ ]:
# OpenAI Generating Model
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:

# Sample 05% of the data randomly
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Sample 5% of the data
sample_size = int(len(df) * 0.05)
sample_indices = np.random.choice(df.index, size=sample_size, replace=False)
df_sample = df.loc[sample_indices].reset_index(drop=True)

print(f"Total data: {len(df)}")
print(f"Sampled data (05%): {len(df_sample)}")
print(f"\nSample data:")
print(df_sample.head())

Total data: 82498
Sampled data (05%): 4124

Sample data:
       num                                               name  \
0  9251120                     maybe.this.time.(2014).eng.1cd   
1  9211589  down.the.shore.s01.e10.and.justice.for.all.(19...   
2  9380845  uncontrollably.fond.s01.e07.heartache.(2016).e...   
3  9301436  screen.two.s13.e04.the.precious.blood.(1996).e...   
4  9408707                          battlebots.(2015).eng.1cd   

                                             content  \
0  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x89\x9a\x...   
1  b'PK\x03\x04\x14\x00\x00\x00\x08\x007\x8f\x99V...   
2  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x8f\x19\x...   
3  b'PK\x03\x04\x14\x00\x00\x00\x08\x00[\xaa\x99V...   
4  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xf4<\x9aV...   

                                        file_content  
0  ï»¿1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch...  
1  1\r\n00:00:09,275 --> 00:00:11,876\r\n¶ Oh, I ...  
2  1\r\n00:00:07,140 --> 00:00:14,220\r\n<i>Timin.

In [15]:
# Data Cleaning: Remove timestamps and unnecessary content
import re

def clean_subtitle_text(text):
    """
    Clean subtitle text by removing:
    1. Timestamps (SRT format: HH:MM:SS,mmm --> HH:MM:SS,mmm)
    2. Timestamps (VTT format: HH:MM:SS.mmm --> HH:MM:SS.mmm)
    3. Sequence numbers (1, 2, 3, etc. at the start of lines)
    4. Extra whitespace and empty lines
    5. HTML tags (if any)
    6. Speaker labels (Name:)
    """
    
    # Remove SRT format timestamps (HH:MM:SS,mmm --> HH:MM:SS,mmm)
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3}\s*-->\s*\d{2}:\d{2}:\d{2},\d{3}', '', text)
    
    # Remove VTT format timestamps (HH:MM:SS.mmm --> HH:MM:SS.mmm)
    text = re.sub(r'\d{2}:\d{2}:\d{2}\.\d{3}\s*-->\s*\d{2}:\d{2}:\d{2}\.\d{3}', '', text)
    
    # Remove sequence numbers at the start of lines (e.g., "1", "2", "3")
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    
    # Remove HTML tags (e.g., <i>, <b>, </i>, </b>)
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove speaker labels followed by colon (e.g., "JOHN:", "MARY:")
    text = re.sub(r'^[A-Z\s]+:\s*', '', text, flags=re.MULTILINE)
    
    # Remove extra whitespace (multiple spaces/tabs)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove extra newlines
    text = re.sub(r'\n+', '\n', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply cleaning to the sampled data
df_sample['file_content_cleaned'] = df_sample['file_content'].apply(clean_subtitle_text)

print("Data Cleaning Complete!")
print(f"\nBefore Cleaning (first 300 chars):")
print(df_sample['file_content'].iloc[0][:300])
print(f"\nAfter Cleaning (first 300 chars):")
print(df_sample['file_content_cleaned'].iloc[0][:300])
print(f"\nCleaning statistics:")
print(f"Original total characters: {df_sample['file_content'].str.len().sum()}")
print(f"Cleaned total characters: {df_sample['file_content_cleaned'].str.len().sum()}")
print(f"Characters removed: {df_sample['file_content'].str.len().sum() - df_sample['file_content_cleaned'].str.len().sum()}")

Data Cleaning Complete!

Before Cleaning (first 300 chars):
ï»¿1
00:00:06,000 --> 00:00:12,074
Watch any video online with Open-SUBTITLES
Free Browser extension: osdb.link/ext

2
00:00:37,328 --> 00:00:39,706
<i>It could've been
just another summer.</i>

3
00:00:40,790 --> 00:00:43,042
<i>But as I set foot on the sand,</i>

4
00:00:43,209 --> 

After Cleaning (first 300 chars):
ï»¿1 Watch any video online with Open-SUBTITLES Free Browser extension: osdb.link/ext It could've been just another summer. But as I set foot on the sand, that summer suddenly felt different. Like it was going to be the summer that would change my life. The summer of freedom. The summer of endless p

Cleaning statistics:
Original total characters: 248841333
Cleaned total characters: 127823011
Characters removed: 121018322


In [16]:
# Document Chunker with overlapping windows
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter with chunk size and overlap
chunk_size = 500  # Token window size (approximate)
chunk_overlap = 100  # Overlap to maintain context between chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Create chunks for all sampled documents
documents = []

for idx, row in df_sample.iterrows():
    subtitle_text = row['file_content_cleaned']  # Use cleaned content
    subtitle_name = row['name']
    subtitle_id = row['num']
    
    # Split the document into chunks
    chunks = text_splitter.split_text(subtitle_text)
    
    # Add each chunk as a document with metadata
    for chunk_idx, chunk in enumerate(chunks):
        documents.append({
            'content': chunk,
            'metadata': {
                'subtitle_id': subtitle_id,
                'subtitle_name': subtitle_name,
                'chunk_index': chunk_idx,
                'total_chunks': len(chunks)
            }
        })

print(f"Total chunks created: {len(documents)}")
print(f"Sample chunk:")
print(f"Content: {documents[0]['content'][:200]}...")
print(f"Metadata: {documents[0]['metadata']}")

Total chunks created: 318578
Sample chunk:
Content: ï»¿1 Watch any video online with Open-SUBTITLES Free Browser extension: osdb.link/ext It could've been just another summer. But as I set foot on the sand, that summer suddenly felt different. Like it ...
Metadata: {'subtitle_id': 9251120, 'subtitle_name': 'maybe.this.time.(2014).eng.1cd', 'chunk_index': 0, 'total_chunks': 143}


In [17]:
print(documents[2])

{'content': 'call me miss. Don\'t pretend to be a gentleman when you\'re clearly not. So, what should I call you? Rude? Snob? Bitch? And you? Douche? Handsome. - Conceited. - Just like you. - Huh? Jerk! - Exactly your type! Leave me alone. Steph! Aha! Steph! I\'ll just call you Tep. Remove the "S" and the "F". By the way, I\'m Tonio. Will you still be here tomorrow? Don\'t leave yet. I\'m going to court you. I chose to walk away from you. But fate had a different plan. Councilorâ\x80¦ We\'re teaching basic English', 'metadata': {'subtitle_id': 9251120, 'subtitle_name': 'maybe.this.time.(2014).eng.1cd', 'chunk_index': 2, 'total_chunks': 143}}


In [18]:
# Store embeddings in ChromaDB
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Convert documents to LangChain Document format
langchain_docs = [
    Document(
        page_content=doc['content'],
        metadata=doc['metadata']
    )
    for doc in documents
]

# Create or connect to ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=google_embed_model,
    collection_name="subtitle_chunks_gemini",
    persist_directory="./chroma_db_gemini"
)

print(f"Vector store created successfully!")
print(f"Collection name: subtitle_chunks_gemini")
print(f"Number of vectors stored: {vectorstore._collection.count()}")
print(f"Persist directory: ./chroma_db_gemini")

KeyError: '_type'

In [ ]:
# ...existing code...
import os
from getpass import getpass
from math import ceil
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.embeddings import OpenAIEmbeddings
import time

# ensure OPENAI key
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

# Create OpenAI embeddings (choose model as needed)
openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Convert your chunks -> LangChain Document objects if not already done
langchain_docs = [
    Document(page_content=doc["content"], metadata=doc["metadata"])
    for doc in documents
]

persist_dir = "./chroma_db_openai"
collection_name = "subtitle_chunks_openai"
batch_size = 200

vectorstore = None
num_batches = ceil(len(langchain_docs) / batch_size)

for i in range(num_batches):
    batch = langchain_docs[i * batch_size : (i + 1) * batch_size]
    try:
        if vectorstore is None:
            # first batch: create collection and persist
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=openai_embed_model,
                collection_name=collection_name,
                persist_directory=persist_dir,
            )
        else:
            # subsequent batches: add documents (vectorstore will embed using configured embedding)
            vectorstore.add_documents(batch)
        # small pause to reduce rate issues
        time.sleep(1)
        print(f"Batch {i+1}/{num_batches} processed.")
    except Exception as e:
        print(f"Error on batch {i+1}: {e}")
        time.sleep(5)  # backoff and continue
        continue

# Persist and report
try:
    vectorstore.persist()
    print("Vector store ready.")
    print("Collection:", collection_name)
    print("Persist dir:", persist_dir)
    print("Stored count (if available):", getattr(vectorstore._collection, "count", lambda: "n/a")())
except Exception:
    print("Done (could not fetch count).")
# ...existing code...

In [ ]:
 def __call__(self, input: Documents) -> Embeddings:
        # Generate embeddings using the Gemini API
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=input,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        return [list(e.values) for e in response.embeddings]

# Store embeddings in ChromaDB
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Convert documents to LangChain Document format
langchain_docs = [
    Document(
        page_content=doc['content'],
        metadata=doc['metadata']
    )
    for doc in documents
]

# Create or connect to ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=google_embed_model,
    collection_name="subtitle_chunks",
    persist_directory="./chroma_db"
)

print(f"Vector store created successfully!")
print(f"Collection name: subtitle_chunks")
print(f"Number of vectors stored: {vectorstore._collection.count()}")
print(f"Persist directory: ./chroma_db")

In [ ]:
# Demonstration with Sample Text Queries
def demo_search(queries, k=5):
    """
    Demo function to test multiple search queries.
    
    Args:
        queries (List[str]): List of search queries
        k (int): Number of results per query
    """
    for query in queries:
        print(f"\n{'='*80}")
        print(f"QUERY: {query}")
        print(f"{'='*80}")
        
        # Search
        results = semantic_search(query, k=k)
        
        # Display results
        for i, result in enumerate(results, 1):
            print(f"\nMatch {i}:")
            print(f"  Similarity: {result['similarity_score']:.4f}")
            print(f"  Subtitle: {result['subtitle_name']}")
            print(f"  Content: {result['content'][:200]}...")
            print(f"  Source: Chunk {result['chunk_index']} (ID: {result['subtitle_id']})")

# Test with sample queries
test_queries = [
    "summer of freedom",
    "love and happiness",
    "fight and battle",
    "mystery and secrets"
]

print("Running Demo Search with Sample Queries...")
demo_search(test_queries, k=3)

In [ ]:
# End-to-End Pipeline: Audio to Search Results
def search_by_audio(audio_path, k=10):
    """
    Complete pipeline: Audio → Text → Embedding → Search
    
    Args:
        audio_path (str): Path to the audio file
        k (int): Number of top results to return
        
    Returns:
        dict: Complete pipeline results
    """
    print("Step 1: Converting audio to text...")
    query_text = audio_to_text(audio_path)
    
    if query_text is None:
        return {"error": "Failed to transcribe audio"}
    
    print("\nStep 2: Preprocessing query...")
    cleaned_query = preprocess_query(query_text)
    print(f"Cleaned query: {cleaned_query[:100]}...")
    
    print("\nStep 3: Searching for similar subtitles...")
    results = semantic_search(cleaned_query, k=k)
    
    return {
        "original_query": query_text,
        "cleaned_query": cleaned_query,
        "results": results,
        "total_results": len(results)
    }

# Alternative: Direct text search (without audio)
def search_by_text(query_text, k=10):
    """
    Search using text query directly.
    
    Args:
        query_text (str): The search query
        k (int): Number of top results to return
        
    Returns:
        dict: Search results
    """
    print("Preprocessing query...")
    cleaned_query = preprocess_query(query_text)
    
    print("Searching for similar subtitles...")
    results = semantic_search(cleaned_query, k=k)
    
    return {
        "original_query": query_text,
        "cleaned_query": cleaned_query,
        "results": results,
        "total_results": len(results)
    }

print("Pipeline functions created successfully!")

In [ ]:
# Semantic Search Function using ChromaDB
def semantic_search(query_text, k=5):
    """
    Search for similar subtitle chunks using semantic similarity.
    
    Args:
        query_text (str): The search query
        k (int): Number of top results to return
        
    Returns:
        List[dict]: List of results with metadata and similarity scores
    """
    # Load the vector store
    vectorstore = Chroma(
        collection_name="subtitle_chunks_gemini",
        embedding_function=google_embed_model,
        persist_directory="./chroma_db_gemini"
    )
    
    # Search using similarity
    results = vectorstore.similarity_search_with_score(query_text, k=k)
    
    # Format results with metadata
    formatted_results = []
    for doc, score in results:
        formatted_results.append({
            'content': doc.page_content,
            'similarity_score': score,
            'metadata': doc.metadata,
            'subtitle_id': doc.metadata.get('subtitle_id'),
            'subtitle_name': doc.metadata.get('subtitle_name'),
            'chunk_index': doc.metadata.get('chunk_index')
        })
    
    return formatted_results

# Test the semantic search
print("Testing Semantic Search...")
query = "the summer of freedom"
results = semantic_search(query, k=5)

print(f"\n{'='*80}")
print(f"Query: '{query}'")
print(f"{'='*80}")
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Similarity Score: {result['similarity_score']:.4f}")
    print(f"  Subtitle: {result['subtitle_name']}")
    print(f"  Content: {result['content'][:150]}...")
    print(f"  Chunk {result['chunk_index']} of {result['metadata']['total_chunks']}")

In [ ]:
# Query Preprocessing and Embedding
from typing import List, Tuple
from sklearn.metrics.pairwise import cosine_similarity

def preprocess_query(text):
    """
    Preprocess query text (same cleaning as subtitles).
    
    Args:
        text (str): Raw query text
        
    Returns:
        str: Cleaned query text
    """
    # Use the same cleaning function
    return clean_subtitle_text(text)

def create_query_embedding(query_text):
    """
    Create embedding for the query text.
    
    Args:
        query_text (str): The search query
        
    Returns:
        np.array: Embedding vector for the query
    """
    # Clean the query
    cleaned_query = preprocess_query(query_text)
    
    # Create embedding using the same model
    query_embedding = openai_embed_model.embed_query(cleaned_query)
    
    return cleaned_query, np.array(query_embedding)

# Test with a sample query
sample_query = "the summer of freedom and endless possibilities"
cleaned, embedding = create_query_embedding(sample_query)
print(f"Original query: {sample_query}")
print(f"Cleaned query: {cleaned}")
print(f"Embedding shape: {embedding.shape}")

In [ ]:
# Audio to Text Conversion using Whisper
import librosa
import numpy as np

def audio_to_text(audio_path):
    """
    Convert audio file to text using OpenAI's Whisper model.
    
    Args:
        audio_path (str): Path to the audio file (supports mp3, wav, m4a, etc.)
    
    Returns:
        str: Transcribed text from the audio
    """
    try:
        import whisper
        
        # Load the Whisper model
        model = whisper.load_model("base")
        
        # Transcribe the audio
        result = model.transcribe(audio_path, language="en")
        transcribed_text = result['text']
        
        print(f"Transcription successful!")
        print(f"Text length: {len(transcribed_text)} characters")
        print(f"Preview: {transcribed_text[:200]}...")
        
        return transcribed_text
    except Exception as e:
        print(f"Error during transcription: {str(e)}")
        return None

# Example usage (you can replace with actual audio file)
# audio_file = "sample_audio.mp3"
# query_text = audio_to_text(audio_file)

In [ ]:
# Install audio processing libraries
!pip install openai-whisper librosa soundfile